In [ ]:
import os
import math
import csv
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict, OrderedDict
from itertools import count
from typing import Dict, Counter, Any, List

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
from Common.Utils import poisson_per_video_requests


In [ ]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 100
    n_nodes: int = 3
    n_users: int = 200
    step_size: float = 10.0
    arrival_rate: float = 10.0  # users per minute
    zipf_alpha: float = 1.0
    n_videos: int = 500
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.25  # 25% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    buffer_capacity: int = 2000
    window_len: int = 3  # LSTM sequence length (history window)
    nb_interval: int = 200  # train every 200 requests

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

In [ ]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    soft_hits
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'soft_hits'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': total_reward,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'soft_hits': soft_hits
        })


In [ ]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.C = self.cfg.cache_size               # paper’s cache capacity (videos)
        self.k = self.env.mec_cache.get_viewport_tile_budget()

        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]

        self.access_order = []
        
        print(f"NetworkAdapter initialized with capacity: {self.C} videos, {self.k} tiles per video")

    def _cache_video(self, vid, bitmap: np.ndarray) -> None:
        bitmap[vid, 0, :, :] = 1

    def _evict_video(self, vid: int, bitmap: np.ndarray) -> None:
        bitmap[vid, :, :, :] = 0
    
    def _evict_tile(self, vid: int, tile: int, bitmap: np.ndarray) -> None:
        bitmap[vid, 1, tile, :] = 0

    def cache_video(self, vid: int):
        vid_idx = self.get_video_cache_idx(vid)

        if vid_idx != -1:
            self.access_order.remove(vid)
            self.access_order.append(vid)

            return

        self.access_order.append(vid)

        if len(self.access_order) > self.C:
            evict_vid = self.access_order.pop(0)
            evict_idx = self.get_video_cache_idx(evict_vid)

            vid_idx = evict_idx
        else:
            vid_idx = self.video_cache_index.index(-1)

        self.video_cache_index[vid_idx] = vid
        self.tile_cache_index[vid_idx] = [-1] * self.k

    def has_video_base_layer(self, vid: int) -> bool:
        return vid in self.video_cache_index

    def get_video_cache_idx(self, vid: int) -> int:
        for idx, v in enumerate(self.video_cache_index):
            if v == vid:
                return idx
        return -1

    def calc_cache_hits(
        self, 
        vid: int, 
        gop: int,
        viewport: list[int], 
    ) -> tuple[int, int, float]:

        vid_idx = self.get_video_cache_idx(vid)

        if gop == 0 and vid_idx == -1:
            return 0, 12, 0.0
        elif gop == 0 and vid_idx != -1:
            return 12, 0, 0.0

        hits = 12
        misses = 0

        tiles_cached = self.tile_cache_index[vid_idx]

        for t_idx in viewport:
            if t_idx in tiles_cached:
                hits += 1
            else:   
                misses += 1

        return hits, misses, 0.0

    def last_sample_replication(
        self, 
        vid: int, 
        gop: int, 
        viewport: list[int],
    ):
        vid_idx = self.get_video_cache_idx(vid)

        for i, tile in enumerate(viewport):
            self.tile_cache_index[vid_idx][i] = int(tile)

    def get_next_user_gop(self, u: int, gop: int, cfg: Config) -> list[int]:
        viewport = self.env.users_env.users_viewport_tiles[u][gop+1]
        return np.array(
            [y * cfg.n + x for x, y in viewport], dtype=int
        )
    
    def reset(self):
        self.video_cache_index = [-1] * self.C
        self.tile_cache_index = [[-1] * self.k for _ in range(self.C)]
        self.access_order = []

        return self.env.reset()
    
    def env_is_done(self) -> bool:
        return self.env.users_env.users_done >= self.env.users_env.n_users

In [ ]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()

    # 2. Initialize Environment
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.zipf_alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    net_adapter = NetworkAdapter(env, cfg)

    for episode in range(cfg.n_episodes):        
        cache_hits = 0
        cache_misses = 0
        soft_hits = 0.0
        total_reward = 0.0
        avg_psnr = []

        obs, info = net_adapter.reset()

        for step in count():

            # Get active users (not finished all GOPs)
            reqs_state = info['users_requests']
            active_users = [
                req for req in reqs_state if req['gop'] < cfg.n_gops
            ]

            # Main Loop. Process each active user request
            for req in reqs_state:
                u, p, v, g, tiles = req['u'], req['p'], req['video'], req['gop'], req["tiles"]
                viewport = req['viewport'] if req['viewport'] is not None else []

                # print(f"-> Step {step}, User {u}, Video: {v}, Gop: {g}, Viewport: {viewport}")

                has_base_layer = net_adapter.has_video_base_layer(v)

                if g == 0:
                    ch, cm, _ = net_adapter.calc_cache_hits(v, g, viewport)

                    net_adapter.cache_video(vid=v)
                elif g > 0 and has_base_layer:
                    ch, cm, _ = net_adapter.calc_cache_hits(v, g, viewport)

                    net_adapter.last_sample_replication(v, g, viewport)
                else:
                    ch, cm, _ = 0, 16, 0.0

                cache_hits += ch
                cache_misses += cm
                soft_hits += float(ch) / (16.0 if g == 0 else 12.0)

                print(
                    f"Step {step}, User {u}, "
                    f"Video: {v}, Gop: {g}, Viewport: {viewport}, "
                )

            # Updates in the state after processing all active users
            reqs_next_state = net_adapter.env.users_env.step(
                None, None
            )

            if net_adapter.env_is_done():
                break

            info = {'users_requests': reqs_next_state}

            # print(f"Step {step}, Active Users: {len(active_users)}")
            # print(f"Request State: {reqs_state}")
            # print(f"Next Request State: {reqs_next_state}")
            print("----------------------------------------------------------------")

        filename = (
            f"lsr_E{cfg.n_episodes}_U{cfg.n_users}_"
            f"g{cfg.gamma}_"
            f"V{cfg.n_videos}_G{cfg.n_gops}_L{cfg.n_layers}_n{cfg.n}_m{cfg.m}_"
            f"cap{cfg.cache_size}_AR{cfg.arrival_rate}_Z{cfg.zipf_alpha}.csv"
        )

        save_training_results(
            # path_='/home/eduardo/Workspace/CacheVideoPredict360/Results',
            path_=r'c:\Users\es25591\Workspace\CacheVideoPredict360\Results',
            filename=filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            soft_hits=soft_hits
        )